# Project 4: Collisions of hard disks

In this notebook, we explore the *hard-disk model*, a classic system in statistical mechanics
consisting of impermeable disks moving freely within a confined space and undergoing elastic
collisions upon contact. With no long-range forces, friction, or external thermal baths,
this model strips physics down to pure geometry and momentum transfer.

Despite its simplicity, it holds an important place in physics history. Pioneered by Ludwig
Boltzmann to lay the foundations of the kinetic theory of gases, the model continues to yield
surprises today, such as its two-dimensional melting transition, which was only fully resolved
in 2011 through advanced computer simulations.

We will simulate it with the algorithm proposed by Alder and Wainwright in 1957, one of the
very first *molecular dynamics* simulations. Instead of moving the disks with small time steps,
it jumps directly from one collision to the next: it is an *event-driven* algorithm.

- **Part 1**: a single disk bouncing in a box, a pinball,
- **Part 2**: watching it move with an animation,
- **Part 3**: the collision between two disks,
- **Part 4**: many disks in a box, the full Alder and Wainwright algorithm,
- **Part 5**: from Newton to Boltzmann, the distribution of the velocities.

All the disks have the same mass, $m = 1$. The cell below defines a few functions to draw and
animate the disks, so that you can focus on the physics and on the algorithm.

In [ ]:
# Common imports
%matplotlib inline
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.collections import EllipseCollection
from IPython.display import HTML

# the random number generator
rng = np.random.default_rng(seed=2026)

# colors of the disks and of the walls
DISK_COLOR, EDGE_COLOR, WALL_COLOR = "#56B4E9", "#0072B2", "#52514e"


def get_positions(system) -> np.ndarray:
    """The positions of the disks of a system, as an array of shape (n_disks, 2)."""
    positions = system.positions if hasattr(system, "positions") else system.position
    return np.array(positions, ndmin=2)  # a copy: the drawing must not change when the disks move


def draw_disks(system, ax: plt.Axes | None = None, box: tuple[float, float] | None = (1.0, 1.0),
               limits: tuple[tuple[float, float], tuple[float, float]] | None = None) -> EllipseCollection:
    """Draw the disks of a system (an object with a radius and position(s) attributes).

    The disks are drawn in the (Lx, Ly) box or without walls if box is None.
    The size of the disks is in data units, so that touching disks really touch.
    """
    ax = ax if ax is not None else plt.gca()
    if box is not None:
        ax.add_patch(plt.Rectangle((0, 0), *box, fill=False, ec=WALL_COLOR, lw=2, zorder=3))
        limits = limits or ((0, box[0]), (0, box[1]))
    diameter = 2 * system.radius
    disks = EllipseCollection(diameter, diameter, 0, units="xy", offsets=get_positions(system),
                              offset_transform=ax.transData, facecolors=DISK_COLOR,
                              edgecolors=EDGE_COLOR, linewidths=1.5)
    ax.add_collection(disks)
    ax.set(xlim=limits[0], ylim=limits[1], aspect="equal")
    ax.set_axis_off()
    return disks


def animate_disks(system, dt: float, n_frames: int = 200, **kwargs) -> HTML:
    """Animate a system with a method step(dt), calling step(dt) between two frames.

    The keyword arguments are passed to draw_disks.
    """
    fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=80)
    disks = draw_disks(system, ax, **kwargs)
    plt.close(fig)  # do not show the static figure as well

    def make_frame(frame: int):
        if frame > 0:  # the first frame shows the initial condition
            system.step(dt)
        disks.set_offsets(get_positions(system))
        return [disks]

    ani = animation.FuncAnimation(fig, make_frame, frames=n_frames, interval=30, blit=True)
    return HTML(ani.to_jshtml())


def random_positions(n_disks: int, radius: float, rng: np.random.Generator, box: float = 1.0) -> np.ndarray:
    """Positions of n_disks non-overlapping disks in the box [0, box]^2, placed one by one."""
    positions = []
    while len(positions) < n_disks:
        candidate = rng.uniform(radius, box - radius, size=2)
        if all(np.sum((candidate - p) ** 2) > (2 * radius) ** 2 for p in positions):
            positions.append(candidate)
    return np.array(positions)

## Part 1: A single pinball disk

Let us start with a single disk of radius $r$ in a box $L_x \times L_y$. It flies in a
straight line, $\vec r(t) = \vec r(0) + \vec v\, t$, until it hits a wall. For an elastic
collision with a vertical wall, $v_x \to -v_x$, and with a horizontal wall, $v_y \to -v_y$.

There are four possible *events*: the collisions with the right, top, left and bottom walls.
The *events table* is an array of four numbers, the times until each of these events. For
instance, if the disk moves to the right ($v_x > 0$), it hits the right wall when its center
reaches $x = L_x - r$, after a time $(L_x - r - x) / v_x$. If it moves to the left, it never
hits the right wall: the corresponding event is impossible.

The event-driven algorithm is then:

1. find the smallest time $t_\mathrm{coll}$ in the events table: this is the next event,
2. move the disk during $t_\mathrm{coll}$, to the point of contact with the wall,
3. reverse the right velocity component and update the events table.

**The `nan` trick.** Impossible events can be marked with `np.nan` (*not a number*). The
function `np.nanargmin` finds the index of the smallest element of an array and ignores
the `nan` entries:

```python
events = np.array([2.2, 5.5, np.nan, np.nan])
np.nanargmin(events)   # 0: the right wall comes first
```

Write a class with the following structure:

```python
class SingleDisk:
    """A disk of radius r bouncing in the box Lx x Ly."""

    def __init__(self, radius: float, box: tuple[float, float],
                 position: np.ndarray, velocity: np.ndarray) -> None:
        ...
        self.events = np.full(4, np.nan)  # right, top, left, bottom
        self.update_events()

    def update_events(self) -> None:
        """Compute the times until the disk hits each of the four walls."""
        ...

    def next_event(self) -> float:
        """Move the disk to the next event, change its velocity and return the elapsed time."""
        ...
```

*Hints*: Store copies of the arrays, `self.position = np.array(position, dtype=float)`, otherwise
the object would modify the arrays of the caller. With the convention right, top, left,
bottom for the walls, `index % 2` tells you which velocity component is reversed.

### Check your code

Create a disk of radius $0.3$ at $(2.5, 2.5)$ with velocity $(1, 0.4)$ in a box of size $5
\times 5$. Print its events table: the right wall should come first, after a time $2.2$.
Call `next_event()` a few times and draw the trajectory, using `draw_disks(disk, box=(5, 5))`
to draw the disk. After every event, the disk should touch a wall. These are exactly the
snapshots shown in the lecture.

## Part 2: Let's watch the disk move

It would be nice to see the disk moving in an animation. But every call to `next_event()` makes
the disk jump directly to the next wall: this is the whole point of the event-driven algorithm,
but a movie made this way would only show the disk stuck to the walls! A movie needs frames at
*regular* times, while the events happen at *irregular* times.

Add a method `step(dt)` to your class, which evolves the system during a fixed time `dt`:

```python
def step(self, dt: float) -> None:
    """Evolve the disk during a time dt, processing all the events on the way."""
    while (next event happens before dt):
        dt -= self.next_event()
    ...  # fly for the remaining time dt, and remember that the events are now closer!
```

There can be several events, or none at all, during `dt`. Once all the events before `dt` have
been processed, the disk flies for the remaining time, and all the times in the events table
decrease by this amount.

Then use the function `animate_disks` defined at the top of the notebook to see the disk move:

```python
disk = SingleDisk(radius=0.3, box=(5, 5), position=(2.5, 2.5), velocity=(1, 0.423))
animate_disks(disk, dt=0.1, n_frames=200, box=(5, 5))
```

*Hint*: To add a method to a class that you already wrote, you can define a subclass,
`class MovingDisk(SingleDisk):`, which only contains the new method `step`. It inherits all the
other methods from `SingleDisk`. Or simply go back to Part 1 and add the method there.

*Optional*: Try a box of size $5 \times 3$ and other velocities. When is the trajectory
periodic?

## Part 3: Two-disk collision

Before we put many disks in the box, let us study the collision of two disks of the same radius
$r$, without any wall. With $\Delta\vec r = \vec r_2 - \vec r_1$ and
$\Delta\vec v = \vec v_2 - \vec v_1$, the disks touch when their distance is $2r$:

\begin{equation}
  \big|\Delta\vec r + \Delta\vec v\, t_\mathrm{coll}\big| = 2r
\end{equation}

This is a quadratic equation $a t^2 + 2 b t + c = 0$, with
$a = |\Delta\vec v|^2$, $b = \Delta\vec r \cdot \Delta\vec v$ and $c = |\Delta\vec r|^2 - 4r^2$.
The time of the first contact is the smallest root

\begin{equation}
  t_\mathrm{coll} = \frac{-b - \sqrt{b^2 - ac}}{a}
\end{equation}

and a collision only takes place if the disks approach each other and do not miss each other:

\begin{equation}
  b = \Delta\vec r \cdot \Delta\vec v < 0 \qquad \text{and} \qquad b^2 - ac \ge 0
\end{equation}

At the collision, the disks exchange momentum along the unit vector
$\hat r = \Delta\vec r / |\Delta\vec r|$ that connects their centers. For equal masses,
conservation of momentum and energy then gives (see the lecture)

\begin{equation}
  \vec v'_1 = \vec v_1 - \Delta\vec w \qquad
  \vec v'_2 = \vec v_2 + \Delta\vec w \qquad
  \text{with} \qquad \Delta\vec w = -\left(\Delta\vec v \cdot \hat r\right) \hat r
\end{equation}

In words, the two disks swap the components of their velocities along $\hat r$.

1. Write a function `collision_time(dr, dv, sigma)` that returns the time until two disks at
   relative position `dr` with relative velocity `dv` are at a distance `sigma` $= 2r$ from each
   other, or `np.nan` if they never touch. *Bonus*: write it so that it also works when `dr` and
   `dv` are arrays of shape `(n, 2)`, for $n$ pairs of disks at once. You will need it in Part 4!
   (*Hint*: `np.sum(dr * dv, axis=-1)` computes the dot products, and `np.where` replaces the
   `if` statements.)

2. Write a class `TwoDisks`, very similar to `MovingDisk`, with attributes `positions` and
   `velocities` (arrays of shape `(2, 2)`, one row per disk), a single event (the collision of
   the two disks), and methods `next_event()` and `step(dt)`.

### Check your code

Take two disks of radius $r = 0.2$ with

\begin{equation}
  \vec r_1 = \begin{pmatrix} 0 \\ 0 \end{pmatrix} \qquad
  \vec v_1 = \begin{pmatrix} 1 \\ 0.5 \end{pmatrix} \qquad
  \vec r_2 = \begin{pmatrix} 3 \\ 0 \end{pmatrix} \qquad
  \vec v_2 = \begin{pmatrix} -0.4 \\ 0.6 \end{pmatrix}
\end{equation}

After a time $t = 2.5$, you should find

\begin{equation}
  \vec r_1 = \begin{pmatrix} 1.863 \\ 0.908 \end{pmatrix} \qquad
  \vec v_1 = \begin{pmatrix} -0.045 \\ -0.061 \end{pmatrix} \qquad
  \vec r_2 = \begin{pmatrix} 2.637 \\ 1.842 \end{pmatrix} \qquad
  \vec v_2 = \begin{pmatrix} 0.645 \\ 1.161 \end{pmatrix}
\end{equation}

Check that the total momentum and the total energy are conserved. Watch the collision with
`animate_disks(disks, dt=0.02, n_frames=150, box=None, limits=((-0.5, 4), (-0.5, 3)))`.

## Part 4: Many disks

You now have all the ingredients to simulate many disks in a box. To keep things simple, all the
disks have the same radius and mass, and the box is the unit square $[0, 1]^2$.

The events table is now a two-dimensional array of shape `(n_disks, n_disks + 4)`: one row per
disk, one column per possible partner plus walls. The first `n_disks` columns are the collisions with the
other disks, and the last four columns the collisions with the walls (right, top, left, bottom).
The element `events[i, j]` is the time until disk `i` hits disk `j` (or wall `j - n_disks`).

```python
class Disks:
    """Hard disks of the same radius and mass in the unit box."""

    def __init__(self, positions: np.ndarray, velocities: np.ndarray, radius: float) -> None:
        ...
        self.events = np.full((self.n_disks, self.n_disks + 4), np.nan)
        for i in range(self.n_disks):
            self.update_events(i)

    def update_events(self, i: int) -> None:
        """Recompute all the events that involve disk i."""
        ...

    def next_event(self) -> float:
        """Move all the disks to the next event, change the velocities and return the elapsed time."""
        ...

    def step(self, dt: float) -> None:
        """Evolve the disks during a time dt."""
        ...
```

- `update_events(i)` recomputes **row and column** `i` of the table: the times of the
  collisions of disk `i` with all the other disks (the table is symmetric, `events[i, j] =
  events[j, i]`) and with the four walls. A disk does not collide with itself: `events[i, i]` is
  `nan`. With the vectorized `collision_time` of Part 3, the times to all the other disks are
  computed in one line, with `dr = self.positions - self.positions[i]`.

- `next_event()` finds the smallest element of the table. On a 2D array, `np.nanargmin` returns
  an index in the *flattened* array, which `np.unravel_index` turns back into a pair of indices:
  ```python
  i, j = np.unravel_index(np.nanargmin(self.events), self.events.shape)
  ```
  All the disks then move during `t_coll` and all the times in the table decrease by `t_coll`.
  Only the velocities of the disks involved in the event change, so only their rows and columns
  need to be recomputed: one disk for a wall collision, two disks for a disk collision.

- `step(dt)` is the same as before.

### Check your code

Start with two disks and a large radius, and compare with your `TwoDisks` class until they hit a
wall. Then create $20$ disks of radius $0.04$ at random positions, with the function
`random_positions` defined at the top of the notebook, and random velocities, e.g.
`rng.uniform(-1, 1, size=(20, 2))`. Animate them! Do the disks ever overlap? Is the energy
conserved?

## Part 5: From Newton to Boltzmann

Our disks obey Newton's laws, nothing else. Statistical mechanics, on the other hand, predicts
that in equilibrium the velocities are distributed according to the Maxwell-Boltzmann
distribution

\begin{equation}
  \rho(\vec v) = \frac{m}{2\pi k_B T}\, e^{-m v^2 / 2 k_B T}
\end{equation}

in two dimensions. Each component, $v_x$ or $v_y$, is a Gaussian of variance $k_B T / m$, and the
speed $v = |\vec v|$ is distributed according to

\begin{equation}
  \rho(v) = \frac{m v}{k_B T}\, e^{-m v^2 / 2 k_B T}
\end{equation}

But what is the temperature of our gas? It is fixed by the total energy, which is conserved:
with two degrees of freedom per disk, equipartition gives

\begin{equation}
  E = \sum_{i=1}^N \frac{1}{2} m v_i^2 = N k_B T
\end{equation}

Let's put this to the test with a very non-thermal initial state: all the disks have the
**same speed** $|\vec v| = 1$, in random directions.

1. Create $100$ disks at random positions, with unit speeds in random directions. What is
   $k_B T$?
2. Run a long simulation and compute the histogram of the speeds, compared with $\rho(v)$.
   With only $100$ disks, a single
   snapshot gives a noisy histogram: accumulate the speeds of the successive frames. Does the
   histogram converge to the Maxwell-Boltzmann distribution?
4. In another simulation, plot the histogram of the speeds after $1$, $2$ and $20$ collisions per disk on average, and
   compare with $\rho(v)$. This time, average over several independent runs.
5. Do the same for the histogram of $v_x$.

*Hints*: `ax.hist(speeds, bins=40, density=True)` plots a normalized histogram, and
`np.histogram(speeds, bins=bins, density=True)` only computes it. In an animation, the bars drawn by
`bars = ax.bar(...)` can be updated with `bar.set_height(height)`.

### Going further (optional)

- **The billiard break**: put the disks at rest on a square lattice and throw a single fast disk
  at them. Animate! How long does it take for the Maxwell-Boltzmann distribution to set in?
- **The pressure**: every time a disk bounces on a wall, it transfers a momentum $2 m |v_\perp|$ to
  the wall. Measure the pressure $P$, the momentum transferred per unit time and per unit length
  of wall, and compare with the ideal gas law $P A = N k_B T$. What happens at high density?